In [1]:
import pandas as pd 
import requests
import os
import json
import numpy as np

## Create a list of player ID's
###### Player's in this list had a roster spot for at least one game in the regular season

In [2]:
# Opening JSON file - play-by-play
f = open(r'C:\Users\Alex\Documents\NHL-Stats\2024-2025\Data\JSON_files\game_data_all_play_by_play.json')
game_data = json.load(f)

In [4]:
#from IPython.display import JSON
#JSON(game_data)

In [3]:
#create empty lists to put data into
player_id = []
position = []

In [4]:
# pull player id data into the list 
for i in range(len(game_data)):
    for j in game_data[i]['rosterSpots']:
        player_id.append(j['playerId'])
        position.append(j['positionCode'])

In [5]:
#create a dataframe with player id and position info
player_id_position = pd.DataFrame(list(zip(player_id, position)),
               columns =['player_id', 'position'])

In [6]:
#drop duplicate player id's - so we have all players who were on an active roster at some point in the season
player_id_position_distinct = player_id_position.drop_duplicates(subset=['player_id'])

In [7]:
# save player id list to a csv file 
#player_id_position_distinct.to_csv('player_id_position.csv', index = False)

## Pull Player Info 

In [2]:
#open player id file into a pandas database
data = pd.read_csv(r'C:\Users\Alex\Documents\NHL-Stats\2024-2025\Data\csv_files\player_id_position.csv') #index col makes sures another index column is not needed

In [3]:
# column to list using Series.tolist() - create a list of only player id's 
player_code_list = data['player_id'].tolist()

In [6]:
all_url = []

for i in range(len(player_code_list)):
    player_code = player_code_list[i]
    player_code_url = 'https://api-web.nhle.com/v1/player/'+str(player_code)+'/landing'
    all_url.append(player_code_url)

In [9]:
#pull the game data from NHL url all
player_data_all = []

for i in range(len(all_url)):
    response = requests.get(all_url[i])
    player_data_all.append(response.json())

In [13]:
#Store the JSON data in a file
#with open("player_data_all.json", "w") as file:
#    json.dump(player_data_all, file)

In [5]:
#open JSON file 
f = open(r'C:\Users\Alex\Documents\NHL-Stats\2024-2025\Data\JSON_files\player_data_all.json')
player_data_all = json.load(f)

In [53]:
#from IPython.display import JSON
#JSON(player_data_all)

### Player General Info

In [22]:
#create empty lists for dataframe 
player_id = []
first_name = []
last_name = []
birth_date = []
is_active = []
position = []
shoots_catches = []
height_inches = []
height_cm = []
weight_lbs = []
weight_kg = []
birth_city = [] 
birth_country = []
draft_year = []
draft_team = []
draft_round = []
draft_pick_in_round = []
draft_overall_pick = []
head_shot = []
hero_image = []

In [23]:
for i in range(len(player_data_all)):
    player_id.append(player_data_all[i]['playerId'])
    first_name.append(player_data_all[i]['firstName']['default'])
    last_name.append(player_data_all[i]['lastName']['default'])
    birth_date.append(player_data_all[i]['birthDate'])
    is_active.append(player_data_all[i]['isActive'])
    position.append(player_data_all[i]['position'])
    shoots_catches.append(player_data_all[i]['shootsCatches'])
    height_inches.append(player_data_all[i]['heightInInches'])
    height_cm.append(player_data_all[i]['heightInCentimeters'])
    weight_lbs.append(player_data_all[i]['weightInPounds'])
    weight_kg.append(player_data_all[i]['weightInKilograms'])
    birth_city.append(player_data_all[i]['birthCity']['default']) 
    birth_country.append(player_data_all[i]['birthCountry'])
    head_shot.append(player_data_all[i]['headshot'])
    hero_image.append(player_data_all[i]['heroImage'])

    try:
        draft_year.append(player_data_all[i]['draftDetails']['year'])
        draft_team.append(player_data_all[i]['draftDetails']['teamAbbrev'])
        draft_round.append(player_data_all[i]['draftDetails']['round'])
        draft_pick_in_round.append(player_data_all[i]['draftDetails']['pickInRound'])
        draft_overall_pick.append(player_data_all[i]['draftDetails']['overallPick']) 
    except KeyError:
        draft_year.append('NaN')
        draft_team.append('NaN')
        draft_round.append('NaN')
        draft_pick_in_round.append('NaN')
        draft_overall_pick.append('NaN') 
        

In [28]:
#create dataframe 
player_info = pd.DataFrame(list(zip(player_id, first_name, last_name, birth_date, is_active, position, shoots_catches,
                            height_inches, height_cm, weight_lbs, weight_kg, birth_city, birth_country, draft_year,
                            draft_team, draft_round, draft_pick_in_round, draft_overall_pick, head_shot, hero_image)),
               columns =['player_id', 'first_name', 'last_name', 'birth_date', 'is_active', 'position', 'shoots_catches',
                            'height_inches', 'height_cm', 'weight_lbs', 'weight_kg', 'birth_city', 'birth_country', 'draft_year',
                            'draft_team', 'draft_round', 'draft_pick_in_round', 'draft_overall_pick', 'head_shot', 'hero_image'])

In [31]:
#create a csv of the player info
#player_info.to_csv('player_info.csv', index = False)

# 2024-2025 Regular Season Stats

## Goalie Info

#### 2024-2025 Goalie Regular Season Stats

In [112]:
#create lists for dataframe creation
player_id = []
games_played = []
wins = []
losses = []
ot_losses = []
gaa = []
save_pctg = []
shut_outs = []

In [113]:
# pull the info for the dataframe
for i in range(len(player_data_all)):
    if player_data_all[i]['position'] == 'G':
        player_id.append(player_data_all[i]['playerId'])
        try:
            games_played.append(player_data_all[i]['featuredStats']['regularSeason']['subSeason']['gamesPlayed'])
            wins.append(player_data_all[i]['featuredStats']['regularSeason']['subSeason']['wins'])
            losses.append(player_data_all[i]['featuredStats']['regularSeason']['subSeason']['losses'])
            ot_losses.append(player_data_all[i]['featuredStats']['regularSeason']['subSeason']['otLosses'])
            gaa.append(player_data_all[i]['featuredStats']['regularSeason']['subSeason']['goalsAgainstAvg'])
            save_pctg.append(player_data_all[i]['featuredStats']['regularSeason']['subSeason']['savePctg'])
            shut_outs.append(player_data_all[i]['featuredStats']['regularSeason']['subSeason']['shutouts'])
        except KeyError:
            games_played.append('NaN')
            wins.append('NaN')
            losses.append('NaN')
            ot_losses.append('NaN')
            gaa.append('NaN')
            save_pctg.append('NaN')
            shut_outs.append('NaN')           

        

In [129]:
#create dataframe
goalie_reg_stats = pd.DataFrame(list(zip(player_id, games_played, wins, losses, ot_losses, gaa, save_pctg, shut_outs)),
               columns =['player_id', 'games_played', 'wins', 'losses', 'ot_losses', 'gaa', 'save_pctg', 'shut_outs'])

In [133]:
#create a csv of the player info
#goalie_reg_stats.to_csv('goalie_reg_stats.csv', index = False)

#### Goalie Career Stats

In [137]:
#create empty lists for dataframe creation
player_id = []
games_played = []
games_started = []
wins = []
losses = []
ot_losses = []
goals = []
assists = []
goals_against = []
gaa = []
pim = []
save_pctg = []
shots_against = []
shutouts = []

In [138]:
# pull the info for the dataframe
for i in range(len(player_data_all)):
    if player_data_all[i]['position'] == 'G':
        player_id.append(player_data_all[i]['playerId'])
        try:
            games_played.append(player_data_all[i]['careerTotals']['regularSeason']['gamesPlayed'])
            games_started.append(player_data_all[i]['careerTotals']['regularSeason']['gamesStarted'])
            wins.append(player_data_all[i]['careerTotals']['regularSeason']['wins'])
            losses.append(player_data_all[i]['careerTotals']['regularSeason']['losses'])
            ot_losses.append(player_data_all[i]['careerTotals']['regularSeason']['otLosses'])
            goals.append(player_data_all[i]['careerTotals']['regularSeason']['goals'])
            assists.append(player_data_all[i]['careerTotals']['regularSeason']['assists'])
            goals_against.append(player_data_all[i]['careerTotals']['regularSeason']['goalsAgainst'])
            gaa.append(player_data_all[i]['careerTotals']['regularSeason']['goalsAgainstAvg'])
            pim.append(player_data_all[i]['careerTotals']['regularSeason']['pim'])
            save_pctg.append(player_data_all[i]['careerTotals']['regularSeason']['savePctg'])
            shots_against.append(player_data_all[i]['careerTotals']['regularSeason']['shotsAgainst'])
            shutouts.append(player_data_all[i]['careerTotals']['regularSeason']['shutouts'])
        except KeyError: 
            games_played.append('NaN')
            games_started.append('NaN')
            wins.append('NaN')
            losses.append('NaN')
            ot_losses.append('NaN')
            goals.append('NaN')
            assists.append('NaN')
            goals_against.append('NaN')
            gaa.append('NaN')
            pim.append('NaN')
            save_pctg.append('NaN')
            shots_against.append('NaN')
            shutouts.append('NaN')            
        

In [152]:
#create dataframe
goalie_career_reg_stats = pd.DataFrame(list(zip(player_id, games_played, games_started, wins, losses, ot_losses, goals,
                                                   assists, goals_against, gaa, pim, save_pctg, shots_against, shutouts)),
               columns =['player_id', 'games_played', 'games_started', 'wins', 'losses', 'ot_losses', 'goals',
                                'assist', 'goals_against', 'gaa', 'pim', 'save_pctg', 'shots_against', 'shutouts'])

In [155]:
#create a csv 
#goalie_career_reg_stats.to_csv('goalie_career_reg_stats.csv', index = False)

## Skater Info

#### 2024-2025 Skater Regular Season Stats

In [14]:
#create empty lists for dataframe creation
player_id = []
games_played = []
goals = []
assists = []
points = []
game_win_goals = []
ot_goals = []
shots = []
shooting_pctg = []
pim = []
plus_minus = []
pp_goals = []
pp_points = []
sh_goals = []
sh_points = []

In [15]:
for i in range(len(player_data_all)):
    if player_data_all[i]['position'] != 'G':
        player_id.append(player_data_all[i]['playerId'])
        games_played.append(player_data_all[i]['featuredStats']['regularSeason']['subSeason']['gamesPlayed'])
        goals.append(player_data_all[i]['featuredStats']['regularSeason']['subSeason']['goals'])
        assists.append(player_data_all[i]['featuredStats']['regularSeason']['subSeason']['assists'])
        points.append(player_data_all[i]['featuredStats']['regularSeason']['subSeason']['points'])
        game_win_goals.append(player_data_all[i]['featuredStats']['regularSeason']['subSeason']['gameWinningGoals'])
        ot_goals.append(player_data_all[i]['featuredStats']['regularSeason']['subSeason']['otGoals'])
        shots.append(player_data_all[i]['featuredStats']['regularSeason']['subSeason']['shots'])
        shooting_pctg.append(player_data_all[i]['featuredStats']['regularSeason']['subSeason']['shootingPctg'])
        pim.append(player_data_all[i]['featuredStats']['regularSeason']['subSeason']['pim'])
        plus_minus.append(player_data_all[i]['featuredStats']['regularSeason']['subSeason']['plusMinus'])
        pp_goals.append(player_data_all[i]['featuredStats']['regularSeason']['subSeason']['powerPlayGoals'])
        pp_points.append(player_data_all[i]['featuredStats']['regularSeason']['subSeason']['powerPlayPoints'])
        sh_goals.append(player_data_all[i]['featuredStats']['regularSeason']['subSeason']['shorthandedGoals'])
        sh_points.append(player_data_all[i]['featuredStats']['regularSeason']['subSeason']['shorthandedPoints'])

In [23]:
#create dataframe
skater_reg_stats = pd.DataFrame(list(zip(player_id, games_played, goals, assists, points, game_win_goals, ot_goals,
                                            shots, shooting_pctg, pim, plus_minus, pp_goals, pp_points, sh_goals, sh_points)),
               columns =['player_id', 'games_played', 'goals', 'assists', 'points', 'game_win_goals', 'ot_goals',
                                'shots', 'shooting_pctg', 'pim', 'plus_minus', 'pp_goals', 'pp_points', 'sh_goals', 'sh_points'])

In [25]:
#create a csv 
#skater_reg_stats.to_csv('skater_reg_stats.csv', index = False)

#### Goalie Career Stats

In [43]:
#create empty lists for dataframe creation
player_id = []
games_played = []
avg_toi = []
goals = []
assists = []
points = []
game_winning_goals = []
ot_goals = []
shots = []
fo_win_pctg = []
pim = []
plus_minus = []
shooting_pctg = []
pp_goals = []
pp_points = []
sh_goals = []
sh_points = []

In [44]:
for i in range(len(player_data_all)):
    if player_data_all[i]['position'] != 'G':
        player_id.append(player_data_all[i]['playerId'])
        games_played.append(player_data_all[i]['careerTotals']['regularSeason']['gamesPlayed'])
        avg_toi.append(player_data_all[i]['careerTotals']['regularSeason']['avgToi'])
        goals.append(player_data_all[i]['careerTotals']['regularSeason']['goals'])
        assists.append(player_data_all[i]['careerTotals']['regularSeason']['assists'])
        points.append(player_data_all[i]['careerTotals']['regularSeason']['points'])
        game_winning_goals.append(player_data_all[i]['careerTotals']['regularSeason']['gameWinningGoals'])
        ot_goals.append(player_data_all[i]['careerTotals']['regularSeason']['otGoals'])
        shots.append(player_data_all[i]['careerTotals']['regularSeason']['shots'])
        pim.append(player_data_all[i]['careerTotals']['regularSeason']['pim'])
        plus_minus.append(player_data_all[i]['careerTotals']['regularSeason']['plusMinus'])   
        pp_goals.append(player_data_all[i]['careerTotals']['regularSeason']['powerPlayGoals'])
        pp_points.append(player_data_all[i]['careerTotals']['regularSeason']['powerPlayPoints'])
        sh_goals.append(player_data_all[i]['careerTotals']['regularSeason']['shorthandedGoals'])
        sh_points.append(player_data_all[i]['careerTotals']['regularSeason']['shorthandedPoints'])

        try: 
            fo_win_pctg.append(player_data_all[i]['careerTotals']['regularSeason']['faceoffWinningPctg'])
            shooting_pctg.append(player_data_all[i]['careerTotals']['regularSeason']['shootingPctg'])
        except KeyError: 
            fo_win_pctg.append('NaN')
            shooting_pctg.append('NaN')

In [49]:
#create dataframe
skater_career_reg_stats = pd.DataFrame(list(zip(player_id, games_played, avg_toi, goals, assists, points, game_winning_goals,
                                            ot_goals, shots, fo_win_pctg, pim, plus_minus, shooting_pctg, pp_goals, 
                                             pp_points, sh_goals, sh_points)),
               columns =['player_id', 'games_played', 'avg_toi', 'goals', 'assists', 'points', 'game_winning_goals', 'ot_goals', 'shots', 
                             'fo_win_pctg', 'pim', 'plus_minus', 'shooting_pctg', 'pp_goals', 'pp_points', 'sh_goals', 'sh_points'])

In [52]:
#create a csv 
#skater_career_reg_stats.to_csv('skater_career_reg_stats.csv', index = False)